In [1]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 8.9 MB/s eta 0:00:00


In [2]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051

In [3]:
import os
from pyngrok import ngrok

In [ ]:
ngrok.kill()

NameError: name 'ngrok' is not defined

In [4]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://reawake-brilliant-favorable.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://reawake-brilliant-favorable.ngrok-free.dev


True

In [5]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

google_search_tool = Tool(
   google_search=GoogleSearch()
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        system_instruction="你是一個中文的AI助手，請用繁體中文回答",
        tools=[google_search_tool],
        response_modalities=["TEXT"],
    )
)

In [6]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [7]:
result2 = stateful_query("校長是誰？")
print(result2)

您好！請問您想詢問哪一所學校或機構的校長？請提供更具體的名稱，我才能為您查詢。


In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit


BODY:  {"destination":"Uecb5c0e1eef83517d8d70393a19967b3","events":[{"type":"message","message":{"type":"text","id":"616516740205249005","quoteToken":"eCzBZ0ewy78blC5jMtPU5n8ORwFHuVUjuuG577tIHEpOPEwxD-vwo4Ug1u9e8w1he4FYklte9wSHbZdDkpry5H6L9y0lMlFMKO8eKLxAlKxY6hH0ys0ozAu959mFUTpRUqFmblhlzHGWa4fauKODAg","markAsReadToken":"F8rDy2CuBv9OeZ4yRK0g1VHjMlqVP92aIeN2OS7XC42F4shbn9MEraz-HEkfpXkRn52VCA20yAwqPGU1mO08VZfQ8LwtRdnYGeC3itvRY7JsIudjM7tjHU5rQ7JDQo2e8ZvfJY8JyB6RCQRix6OtIfVIBTf12OWsTj8KyPCkUCkJsCbgDYNQ0CK98xKj0EBFdx7FTIWLFGnqSsONIPmsyw","text":"AI 介紹明新科技大學，20字之內"},"webhookEventId":"01KT163WJEMG1T8WGFSJGZBR7K","deliveryContext":{"isRedelivery":false},"timestamp":1780303917186,"source":{"type":"user","userId":"Uad9bd22bc745d827475a66a9a23dd568"},"replyToken":"b29ce15e286141a48f7fb480044218e5","mode":"active"}]}


ERROR:__main__:Exception on / [POST]
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 1511, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 919, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
         ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 902, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1240/1799486110.py", line 39, in callback
    handler.handle(body, signature)
  File "/usr/local/lib/python3.12/d

BODY:  {"destination":"Uecb5c0e1eef83517d8d70393a19967b3","events":[{"type":"message","message":{"type":"text","id":"616516758894805338","quoteToken":"hvPg1pj0yL6TqV1F388EXMeC1wWEw4C48k4metnPlPGBTjP7Nfq6ahfwnN5U00yZWkYKUMNCCqG0rurz9b5r9ok0RIn9Jw2rsT66-LuWuv-JT6q-IRSrT-XarswyhLYvwFF8pfUKZK4qUaprWoQLcw","markAsReadToken":"Yu1R7SMeDl8Mn-emnus8lixjhUCbC0rgRWpK86zeFmiGCtjhjb8PARypCoK0-DN7FH2pz5PQR3zVYUHbphmL8FoFiXZ-KZaazY1sOzsV0t58HGH_fxoPCELO3u97kKPM-jazZc_hVeKqtOt0p9lGgIPmlFwFVf8WfkyTw0sQaCttf-Lck3Sqts9y9Y5_7UFpBJf7rJYbInRWssB0wVV_QQ","text":"AI 學校校長是誰"},"webhookEventId":"01KT164794QVXEP6MF31QPC3K3","deliveryContext":{"isRedelivery":false},"timestamp":1780303928145,"source":{"type":"user","userId":"Uad9bd22bc745d827475a66a9a23dd568"},"replyToken":"08722d5117494991935b7586e3f9869b","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [01/Jun/2026 08:52:10] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"Uecb5c0e1eef83517d8d70393a19967b3","events":[{"type":"message","message":{"type":"text","id":"616516781510492394","quoteToken":"8Dkxn3LKEJx4IBiX7p_ut4u829ybSDLw0HEnxTcnAvNmXY4j0lpPAIw5lgjbh56cpc1XkdliPoqgVEeCeAFSO5RF11-zV-xY8fl30aYYV4RSaHKHd4APspp1yW1RCE7X8p08eK5v7sRCyt2yYqyNwA","markAsReadToken":"-DZddyqPJgMxmPMPYK_QxGoY0IGQ_MoLdDOSzbiCzJAkKuIf6mW_rL4EtiwF4jlV88xc4QBwXwYKpC_okePrTxkpT5yXMRfNJY9ru8kXA1FRPp_N_gEVvdqjNQtkLmoKrJtB3IZf7dT6DRmyRkm_V_1Hd8QWVxSI1CfQcASbg3ikpnz9IomNPEXAzqSaIB_NuGEyfEQMPnE3CALNf9wNow","text":"AI 明新科技大學學校校長是誰"},"webhookEventId":"01KT164MJ2HKT8SMET892FMK5E","deliveryContext":{"isRedelivery":false},"timestamp":1780303941758,"source":{"type":"user","userId":"Uad9bd22bc745d827475a66a9a23dd568"},"replyToken":"4f27b627e35047e386491b207b5a8f59","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [01/Jun/2026 08:52:26] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"Uecb5c0e1eef83517d8d70393a19967b3","events":[{"type":"message","message":{"type":"text","id":"616516804630020187","quoteToken":"EDT8t9xYjJHHgl-nuEDy3IHMSfPB64ukZIKI2ArFghIXjO6dDxmRIVxwFj_6mo4vdWTytEhUtjcFITY6fofoDwZU-Syq1jD8tnW5WjhKpQuxs0ryE28CZzs25Z8n4aaktbpHxEnEfGYh3azYyC_I8w","markAsReadToken":"7oeSfesZz22eN22a3Cm7R3i_itbNxolEghVXvOrJ7F4orVUMx7WppVOiMZL-q6mBygAEdRTZOnTZVFnD0xxPPQUTBqmrlVgYqH9QPn7Mn_PFPTOyHY1Q3P0UST5ZosrR5u8HgjrrQG5uN3WAYNmx-jn_5Nj77nPzrekWAcJ6ih2l5DgukfmXvf5WUn2QhhfWnqhok-xPJoiNnn2NTmOzJA","text":"AI 介紹明新科技大學，20字之內"},"webhookEventId":"01KT1651ZZR4NYVSQ9WGQYXDXY","deliveryContext":{"isRedelivery":false},"timestamp":1780303955492,"source":{"type":"user","userId":"Uad9bd22bc745d827475a66a9a23dd568"},"replyToken":"a779e403eb864a60ac6a8967c566e0eb","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [01/Jun/2026 08:52:40] "POST / HTTP/1.1" 200 -
